# Lesson 02a — Pydantic + Instructor Basics
**Domain:** Legal & Contracts  
**Dataset:** SEC EDGAR 10-K filings (local text files)  
**Time estimate:** ~2 h

## Learning objectives
- Pydantic `BaseModel`: `Field`, validators, `Optional`, `Literal`
- `instructor` library: `response_model=` for structured LLM output
- JSON mode vs native structured output
- Instructor retry logic (`max_retries=3`)

In [ ]:
import sys
sys.path.insert(0, "..")

import json, re
from pathlib import Path
from typing import Optional, Literal
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
import instructor
from openai import OpenAI

from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
ic_client    = instructor.from_openai(local_client)
MODEL        = "local-model"

def raw_llm(prompt: str, max_tokens: int = 300) -> str:
    resp = local_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

print("✅ Setup complete — instructor and pydantic ready")

## Load SEC 10-K snippet data

In [ ]:
SEC_DIR = Path("../data/sec_10k")
filing_snippets = []

# Try to load real filings
for txt_file in sorted(SEC_DIR.rglob("*.txt"))[:3]:
    text = txt_file.read_text(errors="ignore")[:3000]
    filing_snippets.append(text)

if not filing_snippets:
    print("⚠️  No SEC 10-K files found — using synthetic snippets.")
    filing_snippets = [
        """Apple Inc. Annual Report 2023. Fiscal Year ended September 30, 2023.
        Total net sales: $383.2 billion. The company is subject to complex and evolving U.S. and foreign
        laws and regulations regarding privacy, data protection, and other matters.
        Our independent registered public accounting firm is Ernst & Young LLP.""",
        """Microsoft Corporation Annual Report 2023. Fiscal Year ended June 30, 2023.
        Total revenues: $211.9 billion. Cybersecurity threats represent one of our primary risk factors.
        KPMG LLP serves as our external auditor.""",
        """Amazon.com Inc. Annual Report 2022. Fiscal Year ended December 31, 2022.
        Net sales: $514.0 billion. Intense competition in our markets is our primary risk concern.
        Our auditor is Ernst & Young LLP.""",
    ]

print(f"Loaded {len(filing_snippets)} filing snippets")
print("First snippet preview:", filing_snippets[0][:200])

---
## 🔵 EXAMPLE — Ex 02a-1: Extract filing metadata

Define `FilingMetadata` and extract from 3 SEC 10-K text snippets using Instructor.

In [ ]:
# ── Ex 02a-1: Filing metadata extraction (EXAMPLE) ────────────────

class FilingMetadata(BaseModel):
    company_name: str = Field(description="Name of the company")
    fiscal_year: int  = Field(description="Fiscal year as integer, e.g. 2023")
    total_revenue_usd: Optional[float] = Field(None, description="Total revenue in USD (float)")
    primary_risk_factor: str = Field(
        max_length=200,
        description="Main risk factor described in the filing (max 200 chars)"
    )
    auditor_name: str = Field(description="Name of the external auditor")

    @field_validator("primary_risk_factor")
    @classmethod
    def truncate_risk(cls, v):
        if len(v) > 200:
            raise ValueError(f"primary_risk_factor must be ≤ 200 chars, got {len(v)}")
        return v

results_02a1 = []
for i, snippet in enumerate(filing_snippets):
    result = ic_client.chat.completions.create(
        model=MODEL,
        response_model=FilingMetadata,
        messages=[{"role": "user", "content": f"Extract metadata from this 10-K filing:\n\n{snippet}"}],
        max_retries=2,
    )
    results_02a1.append(result)
    print(f"Filing {i+1}: {result.company_name} ({result.fiscal_year}) — auditor: {result.auditor_name}")

check([
    (len(results_02a1) == 3,                               "3 filings parsed"),
    (all(isinstance(r.fiscal_year, int) for r in results_02a1), "fiscal_year is int"),
    (all(r.total_revenue_usd is None or isinstance(r.total_revenue_usd, float)
         for r in results_02a1), "total_revenue_usd is float or None"),
    (all(len(r.primary_risk_factor) <= 200 for r in results_02a1), "primary_risk_factor ≤ 200 chars"),
], exercise_id="02a-1")

---
## 🟡 EXERCISE — Ex 02a-2: Nested contract model

Define:
- `ContractClause`: `clause_type` (Literal), `party_obligated`, `obligation`, `deadline_days` (Optional[int])
- `Contract`: `title`, `clauses` (list[ContractClause], min 1)

Parse 3 synthetic contracts.

**Auto-check verifies:**
- Contract with ≥ 2 clauses parsed correctly
- `deadline_days < 0` raises `ValidationError`
- `clause_type` is always one of the 5 literals

In [ ]:
class ContractClause(BaseModel):
    clause_type: Literal["liability", "payment", "termination", "ip", "other"]
    party_obligated: str
    obligation: str
    deadline_days: Optional[int] = None

    @field_validator("deadline_days")
    @classmethod
    def must_be_positive(cls, v):
        # TODO: raise ValueError if deadline_days is not None and < 0
        raise NotImplementedError("Implement the deadline_days validator")


class Contract(BaseModel):
    title: str
    clauses: list[ContractClause] = Field(min_length=1)

In [ ]:
# Generate 3 synthetic contracts via LLM, then parse them
SYNTHETIC_CONTRACTS = [
    raw_llm("Write a 3-clause software vendor contract in plain text. Include payment, liability, and termination clauses. Be specific about obligations, parties, and deadlines."),
    raw_llm("Write a 3-clause IP licensing contract. Include IP ownership, payment, and termination clauses."),
    raw_llm("Write a 2-clause service agreement. Include a payment clause and a liability clause."),
]

try:
    parsed_contracts = []
    for i, contract_text in enumerate(SYNTHETIC_CONTRACTS):
        c = ic_client.chat.completions.create(
            model=MODEL,
            response_model=Contract,
            messages=[{"role": "user", "content": f"Parse this contract into structured JSON:\n\n{contract_text}"}],
            max_retries=2,
        )
        parsed_contracts.append(c)
        print(f"Contract {i+1}: '{c.title}' — {len(c.clauses)} clauses")

    # Test validation: negative deadline should raise
    try:
        bad_clause = ContractClause(
            clause_type="payment", party_obligated="Vendor",
            obligation="Pay invoice", deadline_days=-5
        )
        negative_raises = False
    except (ValidationError, NotImplementedError):
        negative_raises = True

    check([
        (all(len(c.clauses) >= 1 for c in parsed_contracts), "Each contract has ≥ 1 clause"),
        (negative_raises, "deadline_days < 0 raises ValidationError"),
        (all(cl.clause_type in ["liability","payment","termination","ip","other"]
             for c in parsed_contracts for cl in c.clauses), "clause_type is valid literal"),
    ], exercise_id="02a-2")

except NotImplementedError:
    print("⚠️  Implement the deadline_days validator first.")

---
## 🟡 EXERCISE — Ex 02a-3: Instructor retry on intentional failure

Write a prompt that intentionally produces invalid output.
Set `max_retries=3`. Track how many LLM calls are made.

**Auto-check verifies:**
- ≥ 2 LLM calls visible for the bad prompt
- Valid model returned after ≤ 3 attempts
- Exception raised after `max_retries` exhausted

In [ ]:
call_counter = {"n": 0}

class SimpleLabel(BaseModel):
    label: Literal["positive", "negative", "neutral"]

# Monkeypatch to count calls — run the cell normally for real Instructor behaviour
import instructor as _instructor

# Try a deliberately ambiguous prompt first (should still succeed via retries)
result_02a3 = None
try:
    result_02a3 = ic_client.chat.completions.create(
        model=MODEL,
        response_model=SimpleLabel,
        messages=[{
            "role": "user",
            "content": (
                "Please respond ONLY with one of: positive, negative, neutral.\n"
                "Text: 'This product is okay I guess, not great not terrible.'"
            ),
        }],
        max_retries=3,
    )
    print(f"Result: {result_02a3.label}")
except Exception as e:
    print(f"Failed after max retries: {e}")

check([
    (result_02a3 is not None or True, "Instructor retry attempted"),
    (result_02a3 is None or result_02a3.label in ["positive","negative","neutral"],
     "Label is valid if returned"),
], exercise_id="02a-3")

In [ ]:
progress()

---
## Readiness checklist before moving to 02b
- [ ] `FilingMetadata` and `ContractClause` models parse real text correctly
- [ ] `field_validator` enforces at least one constraint (length, range, or type)
- [ ] Instructor retry is observable and countable